# 🏆 Thala_For_A_Reason — Candidate Ranking System

**India Runs Data & AI Challenge** | Intelligent Candidate Discovery & Ranking

This notebook demonstrates the full ranking pipeline for the Redrob AI Senior AI Engineer role.

---

## Pipeline Architecture
1. **Stage 1**: Dense bi-encoder retrieval (FAISS + MiniLM)
2. **Stage 2**: Hard disqualifier & honeypot trap filtering
3. **Stage 3**: Cross-encoder reranking (ms-marco-MiniLM)
4. **Stage 4**: Reciprocal Rank Fusion (RRF)
5. **Stage 5**: Heuristic blending + evidence-based reasoning

## Step 1: Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow pandas numpy

## Step 2: Clone the Repository

Models and pre-computed artifacts are stored via **Git LFS** and will be pulled automatically.

In [ ]:
import os

REPO_URL = "https://github.com/Piyushkp06/INDIA_RUNS_HACKATHON.git"
REPO_DIR = "INDIA_RUNS_HACKATHON"

if not os.path.exists(REPO_DIR):
    !git lfs install
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Step 3: Verify Models & Artifacts

Models (`model/`, `local_cross_encoder/`) and pre-computed artifacts (`.faiss`, `.parquet`) are bundled in the repo via Git LFS.

In [ ]:
required_paths = [
    './model/config.json',
    './local_cross_encoder/config.json',
    './applicant_metadata.parquet',
    './applicant_space.faiss'
]

all_ok = True
for p in required_paths:
    exists = os.path.exists(p)
    size = f'{os.path.getsize(p)/1024/1024:.1f} MB' if exists else 'MISSING'
    status = '✅' if exists else '❌'
    print(f'{status} {p} ({size})')
    if not exists:
        all_ok = False

if all_ok:
    print('\n✅ All models and artifacts present. Ready to rank!')
else:
    print('\n❌ Some files are missing. Check Git LFS: run `git lfs pull`')

## Step 4: Upload candidates.jsonl

The dataset is too large for GitHub (~487 MB). Upload it here or provide a path.

In [ ]:
# Option A: Upload from your machine
from google.colab import files
uploaded = files.upload()  # Upload candidates.jsonl here
CANDIDATES_PATH = './candidates.jsonl'

# Option B: If already available, set the path directly (comment out Option A)
# CANDIDATES_PATH = './Data/India_runs_data_and_ai_challenge/candidates.jsonl'

# Verify
if os.path.exists(CANDIDATES_PATH):
    size_mb = os.path.getsize(CANDIDATES_PATH) / (1024 * 1024)
    print(f'✅ Found candidates file: {size_mb:.1f} MB')
else:
    print(f'❌ File not found at {CANDIDATES_PATH}')

## Step 5: Run the Ranking Pipeline

This runs fully offline using pre-loaded models and pre-computed FAISS index.

In [ ]:
OUTPUT_CSV = './Thala_For_A_Reason.csv'

!python rank.py --candidates {CANDIDATES_PATH} --out {OUTPUT_CSV}

print(f'\n✅ Submission written to {OUTPUT_CSV}')

## Step 6: Validate Submission

In [ ]:
!python Data/India_runs_data_and_ai_challenge/validate_submission.py {OUTPUT_CSV}

## Step 7: Preview Results

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print(f'Total ranked candidates: {len(df)}')
print(f'Score range: {df["score"].min():.4f} — {df["score"].max():.4f}')
print(f'\nTop 10 candidates:\n')
df.head(10)

In [ ]:
# Show reasoning for top 5
for _, row in df.head(5).iterrows():
    print(f"Rank {int(row['rank'])}: {row['candidate_id']} (score: {row['score']:.4f})")
    print(f"  → {row['reasoning']}")
    print()

## Step 8: Download Submission CSV

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)